# 01 — Exploratory Data Analysis

Week 1 deliverable (CLAUDE.md §10). First exploratory pass over the StatsBomb
360 open data:

1. Total 360-available matches (and counts per competition vs. the target scope).
2. Average events / freeze frames per match.
3. Distribution of visible-player counts per freeze frame.
4. First sanity-check render of a freeze frame via `mplsoccer`.

Run top-to-bottom. Network calls hit the free StatsBomb open data (no creds).

In [ ]:
import sys
from pathlib import Path

# Make the repo root importable (so `import data.load` works from notebooks/).
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from data import load

pd.set_option("display.max_rows", 80)
print("repo root:", ROOT)

## 1. 360-available matches

Load the cached match list (`data/processed/match_ids.json`). If it is missing,
build it now (this lists matches for every 360 competition — a few minutes over
the network). Equivalent CLI: `python -m data.load` / `make match-ids`.

_Verified counts (2026-06-08): 426 matches with 360 data; 326 in target scope —
FIFA WC '22 (64), Women's WC '23 (64), Euro '20 / '24 (51 each), Bundesliga
'23/24 (34), Women's Euro '22 / '25 (31 each). Extra 360 comps also exist:
La Liga '20/21, Ligue 1 ×2, MLS '23, AFCON '23._

In [ ]:
try:
    matches = pd.DataFrame(load.load_cached_match_ids())
    print("loaded cached match list")
except FileNotFoundError:
    print("no cache found — building (this calls the network)...")
    matches = load.list_available_matches()
    load.cache_match_ids(matches)

print(f"total 360-available matches: {len(matches)}")
print(f"target-scope matches:        {int(matches['is_target'].sum())}")
matches.head()

In [ ]:
# Counts per competition-season, flagged by whether it is in our target scope.
by_comp = (
    matches.groupby(["competition_name", "season_name", "is_target"])
    .size()
    .reset_index(name="matches")
    .sort_values("matches", ascending=False)
)
display(by_comp)

ax = (
    by_comp.assign(label=by_comp["competition_name"] + " " + by_comp["season_name"])
    .set_index("label")["matches"]
    .sort_values()
    .plot.barh(figsize=(8, 6))
)
ax.set_xlabel("matches with 360 data")
ax.set_title("360-available matches per competition-season")
plt.tight_layout()
plt.show()

## 2. Events / freeze frames per match (sample)

Loading every match is slow, so sample a handful from the target scope and
measure event and freeze-frame volume. Inspect the raw columns first — the 360
frames schema is what later weeks build on.

In [ ]:
SAMPLE_N = 5
pool = matches[matches["is_target"]] if matches["is_target"].any() else matches
sample_ids = pool["match_id"].sample(min(SAMPLE_N, len(pool)), random_state=42).tolist()
print("sampled match_ids:", sample_ids)

rows = []
frames_cache = {}
for mid in sample_ids:
    events = load.load_events(mid)
    frames = load.load_frames(mid)
    frames_cache[mid] = frames
    # One freeze frame = one event id; count distinct event ids in the 360 file.
    id_col = "id" if "id" in frames.columns else frames.columns[0]
    rows.append(
        {
            "match_id": mid,
            "n_events": len(events),
            "n_frames": frames[id_col].nunique(),
            "n_frame_rows": len(frames),
        }
    )

per_match = pd.DataFrame(rows)
display(per_match)
print(per_match[["n_events", "n_frames"]].mean().rename("mean"))
print("\n360 frames columns:", list(next(iter(frames_cache.values())).columns))

## 3. Distribution of visible-player counts per frame

CLAUDE.md §4.3 keeps only frames with ≥ 10 visible players. Check where the
distribution sits and how many frames the ≥ 10 cut would drop.

In [ ]:
counts = []
for _mid, frames in frames_cache.items():
    id_col = "id" if "id" in frames.columns else frames.columns[0]
    counts.append(frames.groupby(id_col).size())
visible_per_frame = pd.concat(counts)

print(visible_per_frame.describe())
frac_ge10 = (visible_per_frame >= 10).mean()
print(f"\nfraction of frames with >= 10 visible players: {frac_ge10:.3f}")

ax = visible_per_frame.plot.hist(bins=range(0, 24), figsize=(8, 4), edgecolor="white")
ax.axvline(10, color="crimson", ls="--", label=">= 10 cut")
ax.set_xlabel("visible players per freeze frame")
ax.legend()
ax.set_title("Visible-player count distribution (sampled matches)")
plt.tight_layout()
plt.show()

## 4. Freeze-frame render via mplsoccer

Pick one event that has a freeze frame and draw the visible players on a
StatsBomb pitch: teammates vs. opponents, with the actor (ball-carrier) and
keeper highlighted. This is the visual primitive the dashboard will build on.

In [ ]:
from mplsoccer import Pitch

mid = sample_ids[0]
frames = frames_cache[mid]
id_col = "id" if "id" in frames.columns else frames.columns[0]

# Choose an event with a well-populated frame.
frame_sizes = frames.groupby(id_col).size().sort_values(ascending=False)
event_id = frame_sizes.index[0]
ff = frames[frames[id_col] == event_id].copy()

# Coordinates: prefer explicit x/y, else split a 'location' [x, y] column.
if {"x", "y"}.issubset(ff.columns):
    ff["_x"], ff["_y"] = ff["x"], ff["y"]
else:
    loc = ff["location"].apply(pd.Series)
    ff["_x"], ff["_y"] = loc[0], loc[1]

pitch = Pitch(pitch_type="statsbomb", line_color="#222222")
fig, ax = pitch.draw(figsize=(10, 7))

team = ff[ff["teammate"]]
opp = ff[~ff["teammate"]]
pitch.scatter(
    team["_x"],
    team["_y"],
    s=160,
    color="#1f77b4",
    edgecolors="white",
    ax=ax,
    label="attacking team",
)
pitch.scatter(
    opp["_x"],
    opp["_y"],
    s=160,
    color="#d62728",
    edgecolors="white",
    ax=ax,
    label="defending team",
)
if "actor" in ff.columns and ff["actor"].any():
    a = ff[ff["actor"]]
    pitch.scatter(
        a["_x"],
        a["_y"],
        s=260,
        color="gold",
        edgecolors="black",
        zorder=5,
        ax=ax,
        label="actor",
    )
if "keeper" in ff.columns and ff["keeper"].any():
    k = ff[ff["keeper"]]
    pitch.scatter(
        k["_x"],
        k["_y"],
        s=160,
        marker="s",
        color="#2ca02c",
        edgecolors="white",
        ax=ax,
        label="keeper",
    )

ax.legend(loc="upper left", bbox_to_anchor=(0, 1.05), ncol=4)
ax.set_title(f"Freeze frame — match {mid}, event {event_id} ({len(ff)} visible)")
plt.show()

---
**Next (Week 2):** lock the four scope decisions with the supervisor (see
`docs/meeting_notes/week1.md`), record the verified match counts above into
`docs/DATA.md`, then build `data/possessions.py`.